# Feature Engineering — EV Service Intelligence

## Objective

Prepare the raw EV service dataset for machine learning.

This notebook will:

- Define the prediction point
- Identify potential data leakage
- Remove features that should not be used for modelling
- Create useful features where required
- Prepare the final ML-ready dataset
- Save the ML-ready dataset for downstream modelling

The output of this notebook will be used by the model development notebooks.

In [4]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv("ev_service_history_raw.csv")

In [6]:
df.head()

,Vehicle_ID,Visit_Number,Purchase_Date,Service_Date,Vehicle_Model,Manufacturing_Date,Vehicle_Age_at_Service,Battery_Age_at_Service,Battery_Health_at_Service,Battery_Replaced,...,Parts_Cost,Labor_Cost,Warranty_Status,Warranty_Covered,Misc_Cost,Repair_Cost,Expected_TAT_Days,Is_Delayed,Actual_Part_Wait_Days,Supplier_Delay_Days
0,EV00002,1,2022-08-01,2023-03-26,TVS iQube,2022-06-16,0.77,0.65,95.68,False,...,0,3120,True,True,100,3220,2.5,False,0,0
1,EV00002,2,2022-08-01,2023-12-15,TVS iQube,2022-06-16,1.50,1.37,90.86,False,...,3013,2300,True,True,0,5313,10.5,False,9,1
2,EV00002,3,2022-08-01,2024-04-18,TVS iQube,2022-06-16,1.84,1.71,88.59,False,...,0,385,True,False,0,385,1.0,False,0,0
3,EV00002,4,2022-08-01,2024-11-05,TVS iQube,2022-06-16,2.39,2.26,84.92,False,...,0,355,True,False,500,855,1.0,False,0,0
4,EV00002,5,2022-08-01,2025-07-05,TVS iQube,2022-06-16,3.05,2.93,80.51,False,...,0,640,False,False,100,740,1.0,True,0,0


## Feature Audit

Each feature will be classified as:

- Keep — available and potentially useful at prediction time
- Drop — unavailable at prediction time / leakage
- Drop — identifier or administrative field with no useful predictive meaning
- Engineer — useful information can be derived from the feature

In [7]:
feature_audit = pd.DataFrame({
    "Feature": df.columns,
    "Data_Type": df.dtypes.astype(str),
    "Unique_Values": df.nunique()
})

feature_audit

,Feature,Data_Type,Unique_Values
Vehicle_ID,Vehicle_ID,object,2615
Visit_Number,Visit_Number,int64,8
Purchase_Date,Purchase_Date,object,1382
Service_Date,Service_Date,object,1775
Vehicle_Model,Vehicle_Model,object,6
Manufacturing_Date,Manufacturing_Date,object,1382
Vehicle_Age_at_Service,Vehicle_Age_at_Service,float64,476
Battery_Age_at_Service,Battery_Age_at_Service,float64,456
Battery_Health_at_Service,Battery_Health_at_Service,float64,1888
Battery_Replaced,Battery_Replaced,bool,2


In [8]:
df.columns.tolist()

['Vehicle_ID',
 'Visit_Number',
 'Purchase_Date',
 'Service_Date',
 'Vehicle_Model',
 'Manufacturing_Date',
 'Vehicle_Age_at_Service',
 'Battery_Age_at_Service',
 'Battery_Health_at_Service',
 'Battery_Replaced',
 'Issue_Family',
 'Exact_Issue',
 'Parts_Required',
 'Parts_Available',
 'Part_Ordered',
 'Expected_Part_ETA_Days',
 'Active_Jobs_On_Arrival',
 'Workshop_Capacity',
 'Workshop_Utilization',
 'Day_Type',
 'Technician_ID',
 'Technician_Experience_Years',
 'Repair_Complexity',
 'Base_Labor_Hours',
 'Technician_Efficiency',
 'Effective_Labor_Hours',
 'Workshop_Wait_Hours',
 'Operational_Time_Hours',
 'Operational_Time_Days',
 'Extra_Operational_Delay_Days',
 'Turnaround_Time_Days',
 'Parts_Cost',
 'Labor_Cost',
 'Warranty_Status',
 'Warranty_Covered',
 'Misc_Cost',
 'Repair_Cost',
 'Expected_TAT_Days',
 'Is_Delayed',
 'Actual_Part_Wait_Days',
 'Supplier_Delay_Days']

In [9]:
df[
    [
        "Base_Labor_Hours",
        "Technician_Efficiency",
        "Effective_Labor_Hours",
        "Workshop_Wait_Hours",
        "Operational_Time_Hours",
        "Operational_Time_Days",
        "Extra_Operational_Delay_Days"
    ]
].head(10)

,Base_Labor_Hours,Technician_Efficiency,Effective_Labor_Hours,Workshop_Wait_Hours,Operational_Time_Hours,Operational_Time_Days,Extra_Operational_Delay_Days
0,6.24,0.91,5.68,3.74,9.42,1.18,1.00
1,4.60,0.94,4.32,0.42,4.74,0.59,0.00
2,0.77,0.97,0.75,0.21,0.96,0.12,0.00
3,0.71,0.88,0.62,2.56,3.18,0.40,0.00
4,1.28,0.88,1.13,1.99,3.12,0.39,1.50
5,2.90,0.94,2.73,5.11,7.84,0.98,2.00
6,2.18,0.94,2.05,1.57,3.62,0.45,0.50
7,1.78,0.94,1.67,0.46,2.13,0.27,0.00
8,1.35,0.88,1.19,0.39,1.58,0.20,0.25
9,2.27,0.97,2.20,1.16,3.36,0.42,0.00


In [10]:
df[
    [
        "Base_Labor_Hours",
        "Effective_Labor_Hours",
        "Operational_Time_Hours",
        "Operational_Time_Days",
        "Extra_Operational_Delay_Days"
    ]
].describe()

,Base_Labor_Hours,Effective_Labor_Hours,Operational_Time_Hours,Operational_Time_Days,Extra_Operational_Delay_Days
count,6583.000000,6583.000000,6583.000000,6583.000000,6583.000000
mean,2.184085,2.023131,3.400603,0.425072,0.280457
std,1.614655,1.479758,2.006503,0.250818,0.416814
min,0.500000,0.440000,0.480000,0.060000,0.000000
25%,1.000000,0.940000,1.800000,0.220000,0.000000
50%,1.480000,1.400000,2.990000,0.370000,0.000000
75%,2.950000,2.750000,4.580000,0.570000,0.500000
max,6.990000,6.990000,12.650000,1.580000,2.000000


## Remove Unusable Features

The following features are removed because they are identifiers,
redundant variables, constant features, or contain information that
would only be available after the prediction point.

In [13]:
drop_columns = [
    "Vehicle_ID",
    "Purchase_Date",
    "Manufacturing_Date",
    "Workshop_Capacity",
    "Technician_ID",
    "Workshop_Wait_Hours",
    "Operational_Time_Hours",
    "Operational_Time_Days",
    "Extra_Operational_Delay_Days",
    "Parts_Cost",
    "Labor_Cost",
    "Misc_Cost",
    "Actual_Part_Wait_Days",
    "Supplier_Delay_Days",
    "Service_Date"
]

In [14]:
ml_ready_df = df.drop(columns=drop_columns)

In [15]:
print("Original shape :", df.shape)
print("ML-ready shape :", ml_ready_df.shape)

Original shape : (6583, 41)
ML-ready shape : (6583, 26)


In [16]:
print(ml_ready_df.columns.tolist())

['Visit_Number', 'Vehicle_Model', 'Vehicle_Age_at_Service', 'Battery_Age_at_Service', 'Battery_Health_at_Service', 'Battery_Replaced', 'Issue_Family', 'Exact_Issue', 'Parts_Required', 'Parts_Available', 'Part_Ordered', 'Expected_Part_ETA_Days', 'Active_Jobs_On_Arrival', 'Workshop_Utilization', 'Day_Type', 'Technician_Experience_Years', 'Repair_Complexity', 'Base_Labor_Hours', 'Technician_Efficiency', 'Effective_Labor_Hours', 'Turnaround_Time_Days', 'Warranty_Status', 'Warranty_Covered', 'Repair_Cost', 'Expected_TAT_Days', 'Is_Delayed']


In [17]:
[col for col in drop_columns if col in ml_ready_df.columns]

[]

In [18]:
ml_ready_df.to_csv("ml_ready.csv", index=False)

In [19]:
target_columns = [
    "Repair_Cost",
    "Turnaround_Time_Days",
    "Is_Delayed"
]

In [20]:
feature_columns = [
    col for col in ml_ready_df.columns
    if col not in target_columns
]

In [21]:
print("Number of Features:", len(feature_columns))
print("Number of Targets :", len(target_columns))

Number of Features: 23
Number of Targets : 3


In [22]:
print("Features:")
print(feature_columns)

Features:
['Visit_Number', 'Vehicle_Model', 'Vehicle_Age_at_Service', 'Battery_Age_at_Service', 'Battery_Health_at_Service', 'Battery_Replaced', 'Issue_Family', 'Exact_Issue', 'Parts_Required', 'Parts_Available', 'Part_Ordered', 'Expected_Part_ETA_Days', 'Active_Jobs_On_Arrival', 'Workshop_Utilization', 'Day_Type', 'Technician_Experience_Years', 'Repair_Complexity', 'Base_Labor_Hours', 'Technician_Efficiency', 'Effective_Labor_Hours', 'Warranty_Status', 'Warranty_Covered', 'Expected_TAT_Days']


In [23]:
print("\nTargets:")
print(target_columns)


Targets:
['Repair_Cost', 'Turnaround_Time_Days', 'Is_Delayed']


In [24]:
print("Feature Engineering Complete")
print(f"ML-ready dataset shape: {ml_ready_df.shape}")
print(f"Features: {len(feature_columns)}")
print(f"Targets: {len(target_columns)}")

Feature Engineering Complete
ML-ready dataset shape: (6583, 26)
Features: 23
Targets: 3
